# This is the Make_sum_more model using Multi-layer perceptron.

In [3]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline


In [5]:
#Read all of the words
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [6]:
len(words)

32033

In [8]:
#Build the vocab of characters and mappings to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [ ]:
#Build the dataset

block_size = 3 #context length, how many characters should we take to predict the next one
X, y = [], []
for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.': #for padding
        ix = stoi[ch]
        X.append(context)
        y.append(ix)
        print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix] #crop and append

X = torch.tensor(X)
y = torch.tensor(y)

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [12]:
X.shape, X.dtype, y.shape, y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

In [ ]:
C = torch.randn((27, 2)) #(rows, columns)

In [31]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [32]:
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [66]:
h = torch.tanh(emb.view(-1, 6) @ W1 + b1)

In [67]:
h

tensor([[-0.9979,  0.9879,  0.9718,  ..., -0.9936,  0.6583, -0.6795],
        [-0.9973,  0.9614,  0.9787,  ..., -0.9939,  0.8371, -0.8320],
        [-0.9995,  0.9921,  0.9538,  ..., -0.9983, -0.5194,  0.1785],
        ...,
        [-1.0000,  1.0000,  0.9294,  ..., -1.0000, -0.7864,  0.8416],
        [-0.3054,  1.0000,  0.7999,  ..., -0.9830,  0.9977,  0.9939],
        [-0.9966, -0.9690,  0.8328,  ...,  0.9933, -0.9915, -0.5904]])

In [68]:
h.shape

torch.Size([32, 100])

In [69]:
W2 = torch.randn(100, 27)
b2 = torch.randn(27)

In [70]:
logits = h @ W2 + b2

In [71]:
counts = logits.exp()

In [72]:
prob = counts / counts.sum(1, keepdims=True)

In [ ]:
prob[0].sum() #adds to one, so it normalized

tensor(1.)

In [77]:
y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

In [81]:
loss = -prob[torch.arange(32), y].log().mean()
loss

tensor(13.9041)